# Module 1 — Ingestion decisions

This notebook documents **why** the ingestion pipeline works the way it does. Each finding follows the same pattern:

- **Question** — what we were unsure about
- **Evidence** — code run against the actual corpus, with its output
- **Decision** — what we chose, and why

**Prerequisites.** The data is not in the repository. To reproduce, run from the repository root:

```bash
uv run python scripts/discover_europe_pmc.py
uv run python scripts/fetch_fulltext.py
```

**Snapshot.** Europe PMC keeps adding papers, so re-running discovery later returns a different set of articles. The numbers below describe the snapshot whose date is printed in the first cell.

**Outputs and licensing.** Outputs are stripped on commit by `nbstripout`, except for cells marked `keep_output`. Kept cells print only counts, licence types, identifiers and structure, never article text. Any new cell that prints article content must not be marked `keep_output`.

In [ ]:
import json
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

from evidence_rag_assistant.ingestion.licences import (
    XLINK_HREF,
    licence_from_url,
    licence_from_xml,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
FULLTEXT = RAW / "fulltext"


def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def local_name(element: ET.Element) -> str:
    return element.tag.rsplit("}", 1)[-1]


def element_text(element: ET.Element) -> str:
    return " ".join(" ".join(element.itertext()).split())


candidates = read_jsonl(RAW / "europe_pmc_candidates.jsonl")
manifest = read_jsonl(RAW / "fulltext_manifest.jsonl")

print(f"Snapshot retrieved: {candidates[0]['retrieved_at'][:10]}")
print(f"Query: {candidates[0]['query']}")
print(f"Candidates: {len(candidates)} | manifest entries: {len(manifest)}")

Snapshot retrieved: 2026-10-20
Query: OPEN_ACCESS:Y AND (TITLE:antibody OR TITLE:"protein structure prediction" OR TITLE:"antibody design")
Candidates: 50 | manifest entries: 50


## Finding 1 — Licences in the search index

**Question.** "Open access" does not mean "free to reuse". How many candidates carry a licence that permits our intended use, and is the topic mix balanced?

**Our intended use.** The public repository contains code and benchmarks that reference articles by identifier. Article text and everything derived from it (chunks, embeddings, indexes) stays on the local machine.

In [ ]:
licences = Counter(c["licence"] or "<missing>" for c in candidates)
eligible = sum(c["licence_ok"] for c in candidates)
structure_prediction = sum("structure prediction" in (c["title"] or "").lower() for c in candidates)

print("Licence according to the search index:", licences.most_common())
print(f"Eligible under our policy: {eligible} / {len(candidates)}")
print(f"Titles mentioning structure prediction: {structure_prediction} / {len(candidates)}")

Licence according to the search index: [('cc by', 34), ('cc by-nc', 8), ('cc by-nc-nd', 8)]
Eligible under our policy: 50 / 50
Titles mentioning structure prediction: 18 / 50


**Decision.**

- Accept all Creative Commons licences, including non-commercial (NC) and no-derivatives (ND) variants. They permit local, non-commercial use, and nothing derived from the text is published.
- Exclude any article whose licence is missing or unrecognized.
- Treat the search index's licence only as a first filter (see Finding 2).
- The topic mix is acceptable, so no need to split the query by topic.

## Finding 2 — Can the search index's licence be trusted?

**Question.** The search API returns a licence per article, but that is index metadata. The legal statement is the `<license>` element in each article's own JATS XML. Do they agree?

**Approach.** `fetch_fulltext.py` downloads each article's XML into memory, reads the licence from the XML, and writes the file to disk only if that licence is allowed. Every attempt, including failures, is recorded in the manifest. Status `cached` means the file was verified in an earlier run and re-verified now.

In [ ]:
statuses = Counter(e["status"] for e in manifest)
parsed = [e for e in manifest if e["xml_licence"]]
mismatches = [e for e in parsed if not e["licence_match"]]

print("Fetch status:", dict(statuses))
print(f"Licences parsed from XML: {len(parsed)}")
print(f"Mismatches among parsed licences: {len(mismatches)}")
for e in manifest:
    if e["xml_licence"] is None:
        print(
            f"Unverified: {e['pmcid']} (index says {e['metadata_licence']!r}, status {e['status']})"
        )

Fetch status: {'cached': 49, 'licence_unverified': 1}
Licences parsed from XML: 49
Mismatches among parsed licences: 0
Unverified: PMC13576444 (index says 'cc by-nc-nd', status licence_unverified)


**Decision.**

- The article's own XML is authoritative; the index licence is not used for the final decision.
- No mismatches among the parsed licences: for this corpus, the index metadata is reliable wherever we could check it. Note the limit of that claim: unverified articles cannot be compared.
- Store licence evidence (the matched URL and licence text) and a SHA-256 hash per article, so every corpus document can be traced back to its source and licence.

## Finding 3 — Publishers declare licences differently (the PLOS bug)

**Question.** The first version of the parser rejected 6 of 50 articles as `licence_unverified`. Were those articles really missing a licence?

**Evidence.** Inspecting the `<permissions>` block of a rejected PLOS article showed the licence URL sitting in an attribute of a nested element:

```xml
<license><license-p>... the <ext-link xlink:href="https://creativecommons.org/licenses/by/4.0/">
Creative Commons Attribution License</ext-link> ...</license-p></license>
```

The first parser only checked the `<license>` element's own `xlink:href` and its text. `itertext()` returns text only, never attributes, so the URL was invisible.

Below, minimal reproductions of the patterns found in the corpus, run through the original parser and the current one. These are hand-written snippets, not article text.

In [ ]:
NAMESPACES = (
    'xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:ali="http://www.niso.org/schemas/ali/1.0/"'
)
URL = "https://creativecommons.org/licenses/by/4.0/"

PATTERNS = {
    "URL in prose (e.g. Oxford)": (
        f"<license><license-p>Distributed under CC BY ({URL}).</license-p></license>"
    ),
    "ali:license_ref (e.g. MDPI, ACS)": (
        f"<license><ali:license_ref>{URL}</ali:license_ref>"
        "<license-p>Open access.</license-p></license>"
    ),
    "Nested ext-link (e.g. PLOS)": (
        f'<license><license-p>Under the <ext-link xlink:href="{URL}">'
        "Creative Commons Attribution License</ext-link>.</license-p></license>"
    ),
    "Abbreviation only (e.g. ACS)": (
        "<license><license-p>This article is licensed under CC-BY-NC-ND 4.0</license-p></license>"
    ),
}


def original_parser(root: ET.Element) -> str | None:
    """First version: checks only the element's own href and its text."""
    for element in root.iter():
        if local_name(element) not in {"license", "license_ref"}:
            continue
        for candidate in (element.get(XLINK_HREF) or "", element_text(element)):
            licence = licence_from_url(candidate)
            if licence:
                return licence
    return None


print(f"{'Pattern':<36} {'original':<12} current")
for label, snippet in PATTERNS.items():
    root = ET.fromstring(f"<article {NAMESPACES}><permissions>{snippet}</permissions></article>")
    current, _ = licence_from_xml(root)
    print(f"{label:<36} {str(original_parser(root)):<12} {current}")

Pattern                              original     current
URL in prose (e.g. Oxford)           cc by        cc by
ali:license_ref (e.g. MDPI, ACS)     cc by        cc by
Nested ext-link (e.g. PLOS)          None         cc by
Abbreviation only (e.g. ACS)         None         None


**Decision.**

- Search the whole `<license>` subtree for `xlink:href` attributes, not just the element itself. Re-running the fetch recovered 5 of the 6 rejected articles.
- Each pattern became a regression test in `tests/test_licences.py`, so the bug cannot silently return.
- Lesson: a standard (JATS) defines the structure, but publishers fill it in differently. Rules must be checked against real files.

## Finding 4 — Licences stated only as an abbreviation

**Question.** The one remaining rejected article (an ACS paper) states its licence only as text: *"This article is licensed under CC-BY-NC-ND 4.0"*, with no URL anywhere. Other ACS articles in the corpus use the same sentence but also include an `<ali:license_ref>` URL. Should the parser accept abbreviations?

**Evidence.** See the last row of the table in Finding 3: neither parser recognizes it, by design.

**Decision.** Keep excluding it. The parser accepts only machine-readable Creative Commons URLs.

- **Why:** one article out of 50 does not justify a wider rule, and every rule added to a licence check must be tested and defended. When unsure, the parser fails closed: a false negative costs one article, a false positive puts content of uncertain status into the corpus.
- **Revisit if** a future source uses abbreviations widely.

## Finding 5 — JATS layout in practice: back matter inside `<body>`

**Question.** JATS defines three parts: `<front>` (metadata, abstract), `<body>` (the paper), `<back>` (references, acknowledgments, funding). Can the extractor simply take the abstract plus `<body>` and skip `<back>`?

**Evidence.** The structure of one article (DyAb, PMC13531962). Paper-specific section titles are replaced by `<content section>`; only back-matter titles are printed.

In [ ]:
BACK_MATTER_WORDS = (
    "acknowledg",
    "funding",
    "disclosure",
    "competing",
    "conflict",
    "availability",
    "supplementary",
    "references",
    "associated data",
    "author contributions",
)

root = ET.parse(FULLTEXT / "PMC13531962.xml").getroot()
meta = root.find("front/article-meta")
body = root.find("body")
assert body is not None

print("Top-level parts:", [local_name(c) for c in root])
print("Abstract in front/article-meta:", meta is not None and meta.find("abstract") is not None)
print("Paragraphs directly in <body>:", sum(local_name(c) == "p" for c in body))
print("\nTop-level <body> sections, in order:")
for sec in body.findall("sec"):
    title = sec.find("title")
    label = element_text(title).lower() if title is not None else ""
    is_back_matter = any(word in label for word in BACK_MATTER_WORDS)
    print("  " + (label if is_back_matter else "<content section>"))

Top-level parts: ['front', 'body']
Abstract in front/article-meta: True
Paragraphs directly in <body>: 3

Top-level <body> sections, in order:
  <content section>
  <content section>
  <content section>
  <content section>
  <content section>
  supplementary material
  acknowledgments
  funding statement
  disclosure statement
  data and materials availability
  supplementary material
  references
  associated data


**Finding.** This article has no `<back>` at all. References, acknowledgments, funding and disclosures are `<sec>` elements inside `<body>`, after the content sections. Taking all of `<body>` would put the reference list into the corpus, where its titles match questions very well and could be cited as evidence.

**Decision.** The extractor cannot decide what to keep by *where* a section sits; it must decide by *what* it is. Signals, from most to least reliable:

1. What the section contains (`<ref-list>`, `<supplementary-material>`)
2. The `sec-type` attribute, where publishers use it
3. The section title, as a fallback

Which signals are needed depends on the whole corpus, so Finding 6 surveys all files.

## Finding 6 — Section survey across the corpus

**Question.** Across all stored articles: how many have a `<back>`, how many have untitled paragraphs directly in `<body>`, which `sec-type` labels are used, and which section titles recur?

Only titles appearing in two or more files are printed. Recurring titles are almost always generic ("Discussion", "References"); paper-specific titles appear once and are almost always real content.

In [ ]:
files = sorted(FULLTEXT.glob("*.xml"))
titles: Counter[str] = Counter()
sec_types: Counter[str] = Counter()
has_back = 0
direct_p = 0

for path in files:
    root = ET.parse(path).getroot()
    if root.find("back") is not None:
        has_back += 1
    body = root.find("body")
    if body is None:
        continue
    if any(local_name(c) == "p" for c in body):
        direct_p += 1
    for sec in body.findall("sec"):
        title = sec.find("title")
        titles[element_text(title).lower() if title is not None else "<untitled>"] += 1
        if sec.get("sec-type"):
            sec_types[sec.get("sec-type")] += 1

print(f"Files: {len(files)}")
print(f"With <back>: {has_back}")
print(f"With paragraphs directly in <body>: {direct_p}")
print("\nsec-type attributes:", sec_types.most_common())
print("\nTop-level section titles appearing in 2+ files:")
for title, n in titles.most_common():
    if n >= 2:
        print(f"{n:3d}  {title[:70]}")
print(f"\n(plus {sum(1 for n in titles.values() if n == 1)} titles that appear only once)")

Files: 49
With <back>: 25
With paragraphs directly in <body>: 4

sec-type attributes: [('ref-list', 23), ('ack', 22), ('associated-data', 19), ('supplementary-material', 13), ('contrib-info', 12), ('intro', 9), ('fn-group', 7), ('discussion', 7), ('conclusions', 5), ('results', 5), ('conclusion', 3), ('methods', 2), ('COI-statement', 2), ('materials|methods', 2), ('introduction', 1), ('glossary', 1), ('discussion | conclusions', 1)]

Top-level section titles appearing in 2+ files:
 29  introduction
 23  references
 22  discussion
 22  supplementary material
 21  results
 19  associated data
 14  author contributions
 13  data availability
 12  contributor information
 12  materials and methods
 11  conclusion
 10  funding
 10  acknowledgments
  9  methods
  8  acknowledgements
  8  supplementary information
  7  footnotes
  7  funding statement
  6  1. introduction
  6  1 introduction
  6  supporting information
  4  conflict of interest
  4  results and discussion
  3  2 materials and

**Finding.** 24 of 49 files have no `<back>`; their back matter sits inside `<body>`. 4 files have untitled paragraphs directly in `<body>`. `sec-type` labels for back matter are common (`ref-list`, `ack`, `associated-data`, `supplementary-material`, `contrib-info`, `fn-group`, `COI-statement`), but labels for content (`intro`, `results`) are rare. Titles for the same kind of section vary in spelling and numbering.

**Decision.**

- **Keep:** title and abstract; untitled paragraphs directly in `<body>` (empty section path); every non-excluded `<body>` section with its section path, including Methods; figure captions.
- **Skip:** all of `<back>`, and any `<body>` section identified as non-content by, in order: its contents (`<ref-list>`, or only `<supplementary-material>`), its `sec-type`, or its normalized title (lowercase, leading numbering removed) on an exact-match list built from this survey. Skipping a section skips its sub-sections.
- **Audit:** every skipped section is logged with its reason, so exclusions can be reviewed.
- **Known simplifications:** data availability statements and appendices are skipped for now; revisit once the benchmark can measure their effect.

In [ ]:
root = ET.parse(FULLTEXT / "PMC13531962.xml").getroot()
body = root.find("body")
for p in body:
    if local_name(p) == "p":
        print(element_text(p)[:200], "...\n")

## Finding 7 — Inline markup and citation markers

**Question.** Are the untitled paragraphs directly in `<body>` really an introduction, and does the text come out clean?

**Evidence.** Reading the 3 untitled paragraphs of PMC13531962 locally (output not kept) confirmed they are the introduction: background, prior work, and the paper's approach. The same output showed two text problems, reproduced below on a hand-written snippet:

- Words split by inline formatting, e.g. `<bold>Dy</bold>ad` printed as "Dy ad", because text fragments were joined with spaces.
- Citation markers (`<xref ref-type="bibr">`) left as stray numbers, e.g. "worldwide, 1 , 2 data".

In [ ]:
snippet = ET.fromstring(
    '<p>Widely used drugs,<xref ref-type="bibr" rid="b1">1</xref>,'
    '<xref ref-type="bibr" rid="b2">2</xref> as shown by '
    "<bold>Dy</bold>ad models.</p>"
)

joined_with_spaces = " ".join(" ".join(snippet.itertext()).split())
joined_without = " ".join("".join(snippet.itertext()).split())

print("Joined with spaces:   ", repr(joined_with_spaces))
print("Joined without spaces:", repr(joined_without))

Joined with spaces:    'Widely used drugs, 1 , 2 as shown by Dy ad models.'
Joined without spaces: 'Widely used drugs,1,2 as shown by Dyad models.'


**Decision.**

- Keep untitled `<body>` paragraphs, with an empty section path.
- Join text within a block element (paragraph, title, caption) without separators, so inline formatting cannot split words; separate block elements from each other.
- Drop `<xref ref-type="bibr">` elements during parsing, then tidy leftover punctuation. Do not remove numbers with a regex afterwards, which cannot tell a citation "7" from "7 mutations".

## Finding 8 — Auditing the extraction

**Question.** Applied to the whole corpus, do the extraction rules keep the content, drop the back matter, and leave nothing unexplained?

**Approach.** `scripts/extract_text.py` writes every extracted block to `data/processed/blocks.jsonl` and every skipped element, with its reason, to `data/processed/extraction_skips.jsonl`. Before extracting a file, it re-checks its SHA-256 against the fetch manifest, so only the exact verified file is used.

In [ ]:
blocks = read_jsonl(RAW.parent / "processed" / "blocks.jsonl")
skips = read_jsonl(RAW.parent / "processed" / "extraction_skips.jsonl")


def skip_signal(reason: str) -> str:
    """Group skip reasons by the signal that triggered them."""
    if reason in {"contains_ref_list", "only_supplementary_material"}:
        return "contents"
    if reason.startswith("sec_type:"):
        return "sec-type"
    if reason.startswith("title:"):
        return "title"
    return reason


print("Articles:", len({b["pmcid"] for b in blocks}))
print("Blocks by kind:", dict(Counter(b["kind"] for b in blocks)))
print(f"Total words: {sum(len(b['text'].split()) for b in blocks):,}")
print("\nSkips by signal:")
for signal, n in Counter(skip_signal(s["reason"]) for s in skips).most_common():
    print(f"  {n:4d}  {signal}")
print("\nUnhandled elements:", sum(s["reason"] == "unhandled_element" for s in skips))
print("Blocks containing LaTeX preambles:", sum("documentclass" in b["text"] for b in blocks))

Articles: 49
Blocks by kind: {'title': 49, 'abstract': 90, 'paragraph': 2621, 'caption': 236}
Total words: 306,955

Skips by signal:
   149  in_back
   103  table_not_extracted
    90  formula_not_extracted
    83  title
    66  sec-type
    49  contents
    12  typed_abstract
     1  no_body

Unhandled elements: 0
Blocks containing LaTeX preambles: 0
